# 04 — Joint low-flow analysis, two gauges, annual minima

Port of `copula.R`. Six candidate copulas fitted by maximum pseudo-likelihood on ranks, ranked by AIC with a bootstrap Cramer-von Mises test, with AND / OR / Kendall joint return periods and the most-likely realisation on each AND curve.

In [ ]:
# --- 1. Which interpreter is this, and does it have the stack? -------------
# A "ModuleNotFoundError: No module named 'numpy'" means this kernel is not the
# environment you installed into. The pip line printed below targets THIS
# kernel, which is what you want -- a bare `pip install` may hit a different one.
import importlib.util, sys

print("interpreter :", sys.executable)
print("version     :", sys.version.split()[0])

REQUIRED = ["numpy", "scipy", "pandas", "matplotlib", "statsmodels",
            "openpyxl", "lmoments3"]
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]

if missing:
    print("\nMISSING:", ", ".join(missing))
    print("\nInstall into THIS kernel by running the following in a terminal")
    print("(the quotes matter -- the path may contain spaces):\n")
    print(f'  "{sys.executable}" -m pip install ' + " ".join(missing))
    print("\nOr from inside this notebook, in a new cell:\n")
    print("  %pip install " + " ".join(missing))
    print("\nIf you use conda, prefer:")
    print("  conda install -n <env> " + " ".join(m for m in missing if m != "lmoments3"))
    print("  conda run -n <env> pip install lmoments3")
else:
    print("\nAll dependencies present.")

In [ ]:
# --- 2. Locate the lowflow package ----------------------------------------
# Searches this directory and its parents for lowflow/__init__.py, so the
# notebook works whether it sits beside the package or in a notebooks/
# subfolder. Set PKG_ROOT by hand if your layout is unusual.
import os, sys, pathlib

PKG_ROOT = None          # e.g. pathlib.Path("/path/to/lowflow-frequency-analysis")

if PKG_ROOT is None:
    here = pathlib.Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "lowflow" / "__init__.py").exists():
            PKG_ROOT = cand
            break

if PKG_ROOT is None:
    raise FileNotFoundError(
        "Could not find lowflow/__init__.py in this directory or any parent.\n"
        "Set PKG_ROOT above to the folder that CONTAINS the lowflow/ directory, "
        "e.g. pathlib.Path('/path/to/lowflow-frequency-analysis')"
    )

PKG_ROOT = pathlib.Path(PKG_ROOT).resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))
os.chdir(PKG_ROOT)       # so relative data/ and out/ paths resolve

import lowflow
print("lowflow package :", lowflow.__file__)
print("working dir     :", os.getcwd())
print("version         :", lowflow.__version__)

In [ ]:
import numpy as np
import pandas as pd

from lowflow import read_flow, block_minima
from lowflow import copulas as cop
from lowflow.marginals import compare, fit_all
import lowflow.figures as figs

figs.apply_style()
pd.set_option("display.width", 170, "display.max_columns", 40)

## Orientation — read this once

`u` is a **droughtiness level**: the probability transform of the *negated* minimum, `u = 1 - F_Q(q)`. So large `u` means low discharge, and the joint drought event is the **upper** set:

| event | meaning | probability |
|---|---|---|
| AND | both rivers below their thresholds | `1 - u - v + C(u,v)` |
| OR | at least one below its threshold | `1 - C(u,v)` |
| Kendall | the whole region at least as rare as the critical layer | `P(Chat(U,V) <= p)` |

Consequently a copula with **upper**-tail dependence — Gumbel, Joe, survival Clayton — is the one that represents simultaneous drought. Plain Clayton on these variables is the wrong tail: it imposes asymptotic independence in joint drought. It is included below so that assumption is tested rather than assumed.

In [ ]:
# Point these at your own workbooks; the two columns must be date and flow.
NAME_X, PATH_X = "Station A", "data/station_a.xlsx"
NAME_Y, PATH_Y = "Station B", "data/station_b.xlsx"
SEASON  = None
D       = 7
MIN_COV = 0.90
T       = (2, 5, 10, 20, 50, 100)
B_GOF   = 199        # raise to 999 for publication
FAMILIES = ("Gumbel", "Joe", "Survival Clayton")   # upper-tail dependent on the
# droughtiness scale, so these are the only admissible candidates. Gaussian,
# Frank and plain Clayton all have lambda_drought = 0; add them back only as a
# one-off sensitivity check, never as a candidate set.

ra = read_flow(PATH_X, name=NAME_X, D=D, zero_policy="keep")
rb = read_flow(PATH_Y, name=NAME_Y, D=D, zero_policy="keep")
print(ra.summary()); print(); print(rb.summary())

In [ ]:
print(ra.daily.head())
print(rb.daily.head())
print(ra.daily.dtypes)
print(rb.daily.dtypes)

In [ ]:
ra.daily.index = ra.daily.index.tz_localize(None)

In [ ]:
print(ra.daily.index[:3])
print(rb.daily.index[:3])

## 1. Pair the records on common days

Only days with a valid D-day mean at **both** gauges can contribute. The masking below keeps the full calendar span, so the per-block coverage screen still counts calendar days — restricting to the common days alone would make coverage identically 1.0 and silently disable the screen.

In [ ]:
from lowflow.pipeline import _joint_records
ja, jb = _joint_records(ra, rb)
print("days with a valid 7-day mean at BOTH gauges:", ja.diagnostics["n_common_days"])

bma = block_minima(ja, season=SEASON, min_coverage=MIN_COV)
bmb = block_minima(jb, season=SEASON, min_coverage=MIN_COV)
keep = sorted(set(bma.used.year) & set(bmb.used.year))
ua = bma.used.set_index("year").loc[keep]
ub = bmb.used.set_index("year").loc[keep]

pair = pd.DataFrame({
    "year": keep,
    f"min_{NAME_X}": ua["min_flow"].to_numpy(),
    f"date_{NAME_X}": ua["date"].to_numpy(),
    f"min_{NAME_Y}": ub["min_flow"].to_numpy(),
    f"date_{NAME_Y}": ub["date"].to_numpy(),
})
pair["lag_days"] = (pd.to_datetime(pair[f"date_{NAME_Y}"])
                    - pd.to_datetime(pair[f"date_{NAME_X}"])).dt.days
qx = pair[f"min_{NAME_X}"].to_numpy(float)
qy = pair[f"min_{NAME_Y}"].to_numpy(float)

print(f"blocks usable for BOTH rivers: {len(keep)} ({min(keep)}-{max(keep)})")
print(f"median |lag| between the two minima: {pair['lag_days'].abs().median():.0f} days")
print(pair.head(8).to_string(index=False))

## 2. Marginals, each on its own evidence

In [ ]:
marg, best_marg = {}, {}
for nm, q in ((NAME_X, qx), (NAME_Y, qy)):
    f = fit_all(q)
    c = compare(f, B=B_GOF, rng=np.random.default_rng(20240617))
    print(f"\n{nm}:")
    print(c[["distribution", "k", "nllh", "AIC", "dAIC", "A2", "p_value",
             "KS_p_invalid", "notes"]].to_string(index=False))
    marg[nm], best_marg[nm] = f, c.loc[0, "distribution"]
    print(f"  selected: {best_marg[nm]}")

mx = marg[NAME_X][best_marg[NAME_X]]
my = marg[NAME_Y][best_marg[NAME_Y]]

## 3. Dependence, model-free first

Kendall's tau, Spearman's rho, threshold-conditional estimates, and the Caperaa-Fougeres-Genest estimate of the joint-drought tail dependence. Establish this **before** choosing a copula, so there is something to judge the choice against.

In [ ]:
u = cop.pseudo_obs(-qx, -qy)       # large u = deep drought
dep = cop.dependence_summary(u)
print(dep.to_string(index=False))
fig = figs.fig_pseudo_obs(u, names=(NAME_X, NAME_Y), dep=dep,
                          title="Droughtiness pseudo-observations")

## 4. Copulas, ranked by AIC

Maximum pseudo-likelihood on **ranks** — not maximum likelihood on parametric CDF values, which saturates at 0 or 1 whenever a sample extreme falls on a fitted support bound. `Sn` is the Cramer-von Mises statistic with a parametric-bootstrap p-value.

In [ ]:
fits = cop.fit_all_copulas(u, families=FAMILIES)
cmp = cop.compare_copulas(fits, B=B_GOF, rng=np.random.default_rng(7))
print(cmp.to_string(index=False))

best = cmp.loc[0, "copula"]
model = fits[best]
lam_emp = float(dep.loc[dep.measure == "lambda_drought (CFG)", "value"].iloc[0])
print(f"\nselected by AIC: {best} (theta = {model.theta:.4f}, tau = {model.tau:.3f}, "
      f"lambda_drought = {model.lambda_drought:.3f})")
print(f"non-parametric lambda_drought = {lam_emp:.3f} for comparison")

if model.lambda_drought < 0.05 and lam_emp > 0.25:
    print(f"\nCHECK: the selected copula implies lambda_drought = {model.lambda_drought:.2f}"
          " (asymptotic independence in joint drought) while the non-parametric")
    print(f"estimate is {lam_emp:.2f}. At n = {len(u)} the CFG estimator is biased upward,")
    print("so this is not proof of a wrong choice, but the joint design values at long")
    print("return periods depend on it. Compare the runner-up below.")

In [ ]:
fig = figs.fig_copula_density(model, title=f"{best} copula")

## 5. Joint return levels

AND, OR and Kendall are three different definitions of a "T-year joint drought" and give materially different discharges. Whichever you quote has to be named.

In [ ]:
design = cop.and_design_point(model, np.array(T, float), mx, my)
design = design.rename(columns={"x": NAME_X, "y": NAME_Y})
print("AND, at the equal-droughtiness design point u = v:")
print(design.to_string(index=False))

In [ ]:
mld = [cop.most_likely_design_point(model, t, mx, my) for t in T]
mld = [m for m in mld if m]
mtab = pd.DataFrame(mld).rename(columns={"x": NAME_X, "y": NAME_Y})
print("Most-likely realisation on each AND curve:")
print(mtab.drop(columns=["density"]).to_string(index=False))

In [ ]:
from scipy import optimize
or_u = []
for t in T:
    try:
        or_u.append(optimize.brentq(
            lambda z: float(cop.or_probability(model, z, z)) - 1.0 / t, 1e-10, 1 - 1e-10))
    except ValueError:
        or_u.append(np.nan)
or_tab = pd.DataFrame({"T": list(T), "u": or_u})
or_tab[NAME_X] = [float(cop.discharge_from_u(mx, w)) if w == w else np.nan for w in or_u]
or_tab[NAME_Y] = [float(cop.discharge_from_u(my, w)) if w == w else np.nan for w in or_u]
print("OR, at least one river below its threshold (u = v):")
print(or_tab.to_string(index=False))

kc = cop.kendall_critical_level(model, np.array(T, float), rng=np.random.default_rng(11))
print("\nKendall (survival) critical levels:")
print(kc.to_string(index=False))

In [ ]:
curves = pd.concat([cop.and_curve(model, t, mx, my, n=400) for t in T], ignore_index=True)
fig = figs.fig_and_curves(
    curves, design.rename(columns={NAME_X: "x", NAME_Y: "y"}), qx, qy,
    names=(NAME_X, NAME_Y),
    mld=[{**m, "x": m[NAME_X], "y": m[NAME_Y]} for m in mtab.to_dict("records")],
    title=f"Joint AND return levels ({best} copula)",
    subtitle=r"$P(Q_{%s} < q_{%s}\ \mathrm{and}\ Q_{%s} < q_{%s}) = 1/T$"
             % (NAME_X, NAME_X, NAME_Y, NAME_Y))

### Sensitivity to the copula choice

If the runner-up is close on AIC, the spread between these two tables is the honest uncertainty in the joint design point from the dependence model alone.

In [ ]:
second = cmp.loc[1, "copula"]
m2 = fits[second]
d2 = cop.and_design_point(m2, np.array(T, float), mx, my).rename(
    columns={"x": NAME_X, "y": NAME_Y})
comp = design[["T", NAME_X, NAME_Y]].merge(
    d2[["T", NAME_X, NAME_Y]], on="T", suffixes=(f"_{best}", f"_{second}"))
print(f"{best} vs {second} (dAIC = {cmp.loc[1, 'dAIC']:.2f}):")
print(comp.to_string(index=False))

## Write every table and figure to disk

In [ ]:
from lowflow.pipeline import BivariateConfig, run_bivariate

cfg = BivariateConfig(
    path_x=PATH_X, path_y=PATH_Y, name_x=NAME_X, name_y=NAME_Y,
    label=f'{NAME_X} and {NAME_Y}, annual minimum {D}-day mean', outdir='out/nb04_copula_annual', season=SEASON, D=D,
    min_coverage=MIN_COV, zero_policy="keep", T=T, B_gof=B_GOF, families=FAMILIES,
)
res = run_bivariate(cfg)
print(f"\n{len(res['written'])} files written to {res['outdir']}")

In [ ]:
from lowflow.blocks import block_minima

bma = block_minima(ja, season=None, min_coverage=MIN_COV)
bmb = block_minima(jb, season=None, min_coverage=MIN_COV)

print(f"{NAME_X} blocks:", len(bma.used))
print(f"{NAME_Y} blocks:", len(bmb.used))

print(bma.used[["year","min_flow"]].head())
print(bmb.used[["year","min_flow"]].head())

keep = sorted(set(bma.used.year) & set(bmb.used.year))
print("Common years:", len(keep))
print(keep)

In [ ]:
print(bma.used.columns)
print(bmb.used.columns)
print(bma.used.isna().sum())
print(bmb.used.isna().sum())

In [ ]:
ua = bma.used.set_index("year").loc[keep]
ub = bmb.used.set_index("year").loc[keep]
print(len(ua))
print(len(ub))
print(ua.head())
print(ub.head())

In [ ]:
print("len keep =", len(keep))

ua = bma.used.set_index("year").loc[keep]
ub = bmb.used.set_index("year").loc[keep]

print("len ua =", len(ua))
print("len ub =", len(ub))

print(ua.head())
print(ub.head())

In [ ]:
qx = ua["min_flow"].to_numpy(float)
qy = ub["min_flow"].to_numpy(float)
print("qx size =", len(qx))
print("qy size =", len(qy))
print("NaN in qx =", np.isnan(qx).sum())
print("NaN in qy =", np.isnan(qy).sum())

In [ ]:
pair = pd.DataFrame({
"year": keep,
"qx": qx,
"qy": qy,
"date_x": ua["date"].to_numpy(),
"date_y": ub["date"].to_numpy(),
})

pair["lag_days"] = (
pd.to_datetime(pair["date_y"])
- pd.to_datetime(pair["date_x"])
).dt.days

print(pair.head())
print(pair["lag_days"].describe())

Next: **05 — the same analysis on seasonal minima**.